# <u>Step 6 :- </u> **Tabular Q-Learning (The Bellman Equation)**

**What is Q-Learning?** :- In Step 5, we saw how states and future discounted rewards work. Now our agent learns which action is best in *each individual state* by using the legendary **Bellman Equation**! <br>
**Q-Table ($Q[s][a]$)** :- A table/dictionary storing the expected future return for taking action $a$ in state $s$. <br>
**Temporal Difference Target** :-
$$\text{Target} = R + \gamma \max_{a'} Q(s', a')$$
**Q-Learning Update Formula** :-
$$Q(s, a) \leftarrow Q(s, a) + \alpha \Big[ \big(R + \gamma \max_{a'} Q(s', a')\big) - Q(s, a) \Big]$$

## 1. Environment (1D State World) :-

- **States** : `[0] -> [1] -> [2] -> [3 (Goal)]`
- **Actions** : `['left', 'right']`
- **Step penalty** : `-1` for every move
- **Goal reward** : `+10` upon reaching state `3` (`done = True`)

In [1]:
actions = ['left', 'right']

def step(state, action):
    if action == 'right':
        next_state = state + 1
    else:
        next_state = state - 1

    # Left boundary check
    if next_state < 0:
        next_state = 0

    if next_state == 3:
        reward = 10
        done = True
    else:
        reward = -1
        done = False

    return next_state, reward, done

## 2. Q-Table & Hyperparameters :-

- `Q[state][action]` stores estimated values for each state-action pair.
- $\alpha = 0.1$ (Learning Rate)
- $\gamma = 0.9$ (Discount Factor)
- $\epsilon = 0.1$ (Exploration Rate)

In [2]:
import random

Q = {
    0 : {'left': 0.0, 'right': 0.0},
    1 : {'left': 0.0, 'right': 0.0},
    2 : {'left': 0.0, 'right': 0.0},
    3 : {'left': 0.0, 'right': 0.0},
}

alpha = 0.1
gamma = 0.9
epsilon = 0.1

## 3. Action Selection ($\epsilon$-Greedy Policy) :-

With probability $\epsilon$, explore randomly. Otherwise, exploit the best action in the current state.

In [3]:
def choose_action(state, Q, epsilon=0.1):
    if random.random() < epsilon:
        return random.choice(actions)
    else:
        if Q[state]['right'] > Q[state]['left']:
            return 'right'
        elif Q[state]['left'] > Q[state]['right']:
            return 'left'
        else:
            return random.choice(actions)  # Break tie randomly if equal

## 4. Q-Learning Training Loop :-

### Inside each step :-
1. **Select Action** using `choose_action(state, Q, epsilon)`
2. **Interact with Environment** : Get `next_state`, `reward`, `done`
3. **Compute Best Next Action Value** : If `done`, $\max Q(s') = 0$, else $\max_{a'} Q(s', a')$
4. **Compute Target** : $\text{target} = R + \gamma \max Q(s')$
5. **Update Q-Table** : $Q(s, a) \leftarrow Q(s, a) + \alpha (\text{target} - Q(s, a))$
6. **Advance State** : `state = next_state`

In [4]:
for i in range(200):
    done = False
    state = 0

    while not done:
        action = choose_action(state, Q, epsilon=0.1)
        next_state, reward, done = step(state, action)

        # If terminal state, future value is 0
        if done:
            best_next = 0.0
        else:
            best_next = max(Q[next_state].values())

        # Bellman Target and TD Update
        target = reward + gamma * best_next
        old_val = Q[state][action]
        Q[state][action] = old_val + alpha * (target - old_val)

        state = next_state

print("After training (200 episodes) :")
for key, value in Q.items():
    print(f"State {key}: left = {round(value['left'], 3)}, right = {round(value['right'], 3)}")

After training (200 episodes) :
State 0: left = 3.089, right = 6.2
State 1: left = 2.798, right = 8.0
State 2: left = 1.088, right = 10.0
State 3: left = 0.0, right = 0.0


## 5. Testing the Trained Agent (Greedy / Exploitation Mode) :-

Now we lock exploration (set $\epsilon = 0$) to see if the agent has learned the optimal path from `0 -> 1 -> 2 -> 3`.

In [5]:
print("Running Trained Agent (epsilon = 0) :-")

done = False
state = 0
eval_rewards = []

while not done:
    action = choose_action(state, Q, epsilon=0.0)  # Pure exploitation
    next_state, reward, done = step(state, action)
    eval_rewards.append(reward)

    print(f"State: {state} | Action: {action:<5} -> Next State: {next_state} | Reward: {reward}")
    state = next_state

print(f"\nFinal State reached: {state} (Goal!)")
print("Collected Rewards:", eval_rewards)
print("Total Return:", sum(eval_rewards))

Running Trained Agent (epsilon = 0) :-
State: 0 | Action: right -> Next State: 1 | Reward: -1
State: 1 | Action: right -> Next State: 2 | Reward: -1
State: 2 | Action: right -> Next State: 3 | Reward: 10

Final State reached: 3 (Goal!)
Collected Rewards: [-1, -1, 10]
Total Return: 8


## 6. Understanding the Learned Values :-

Notice how the values propagated backward from the goal:
- At **State 2**: taking `'right'` immediately reaches goal -> Reward = `+10.0`.
- At **State 1**: taking `'right'` moves to state 2 -> Target = `-1 + 0.9 * 10.0 = 8.0`.
- At **State 0**: taking `'right'` moves to state 1 -> Target = `-1 + 0.9 * 8.0 = 6.2`.

The Bellman Equation has successfully backed up the future goal reward to every prior state!